# Inference Acceleration and KV Cache

> The BF16 weights of a 7B model occupy about 14 GB, while a data-center GPU can perform hundreds of trillions of floating-point operations per second. Those numbers suggest generation should be almost instantaneous. In practice, the model often produces only a few dozen tokens per second.
>
> The bottleneck is not raw compute but data movement. This chapter separates one request into its parts: what happens after the prompt arrives, why generation proceeds serially one token at a time, how KV Cache removes repeated computation, and how the cache creates a new memory burden.
>
> We will study four topics:
>
> 1. **Prefill and Decode**: two stages of one request with very different workloads.
> 2. **KV Cache**: previously computed K/V vectors are reused, so Decode processes only the new token.
> 3. **The cost of KV Cache**: the cache itself can dominate memory, motivating GQA and MQA.
> 4. **Memory-bound execution**: Decode is slow because compute units wait for data, not because they cannot calculate fast enough.

Consider a request with a 2,000-token prompt. The model first reads the prompt as a whole, then generates one token at a time. These two phases have very different computation patterns, and most inference optimizations begin by distinguishing them.


## 1. Prefill and Decode

The first phase is **Prefill**. All 2,000 prompt tokens enter the model together, and the model calculates Attention and FFN outputs for every position. This produces large matrix operations that suit a GPU: thousands of tokens run in parallel and can use much of its compute. Prefill ends when the model returns the first output token.

The second phase is **Decode**. Starting with the second output token, only one new token enters the model at each step. A matrix shaped like `2000 x hidden` becomes `1 x hidden`: it is too narrow to saturate the GPU, yet every step still reads the model weights and a growing history. Generating 300 tokens means performing 300 such serial steps.

```text
Prompt tokens -- Prefill --> first token
                              |
                         Decode step
                              |
                         Decode step
                              |
                             ...
```

From the user's perspective, the two phases create two kinds of delay:

```text
send request -------- first token -- tokens stream out -------- finish
             <- TTFT -> <- one token every TPOT ->
```

**TTFT** (Time To First Token) is the wait from sending a request until the first token appears; queuing and Prefill dominate it. **TPOT** (Time Per Output Token) is the average interval between later tokens; Decode dominates it. A long wait before the model begins speaking is a TTFT problem, while slow token streaming is a TPOT problem. Combining both into one latency number hides the actual bottleneck.


In [ ]:
prompt_tokens = 2000
output_tokens = 300

print("Prefill: process", prompt_tokens, "prompt tokens at once (large matrices, high compute utilization)")
print("Decode : then run", output_tokens, "serial steps (one token each, reading all weights)")
print()
print("Key observation: a long prompt mainly increases TTFT; a long output mainly increases TPOT and total latency.")


## 2. The Repeated-Computation Problem

Only one new token arrives at each Decode step, but Attention must compare it with the **entire history**. The new token's Q/K/V depend on its hidden state in the current layer. Without a cache, obtaining that state requires processing the entire prefix again:

```text
Step 1: [A]              -> token1
Step 2: [A, token1]      -> token2   <- A is computed a second time
Step 3: [A, token1, t2]  -> token3   <- A a third time, token1 a second time
```

By the time the model reaches token $N$, the accumulated prefix work is $1 + 2 + \dots + N \approx N^2/2$. The experiment below uses the number of repeatedly processed historical tokens as a proxy for this quadratic growth.


In [ ]:
def repeated_prefix_proxy(n):
    """Without caching, return historical tokens processed while generating n tokens: 1+2+...+n."""
    return n * (n + 1) // 2

for n in [10, 100, 1000]:
    naive = repeated_prefix_proxy(n)
    cached = n
    print(f"N={n:4d}  without cache: {naive:8d} historical tokens | with cache: {cached:5d}")

print()
print("Key observation: without caching, work grows quadratically; 10x the length wastes about 100x the work.")


The key observation is that **the K/V vectors of historical tokens never change**. The K/V values calculated for “Paris” at step 100 are identical at steps 101 and 200: within one sequence, history grows but does not change. Recomputing them is unnecessary. Storing them after the first calculation gives us the next topic.


## 3. How KV Cache Works

**KV Cache stores the Key and Value vectors already produced by Attention in GPU memory, allowing later steps to reuse them instead of recomputing them.** In plain language, it keeps one copy of each token's “memory” for direct lookup when a new token arrives.

With a KV Cache, each Decode step performs only two new pieces of work: calculate the new token's Q, K, and V, then attend with the new Q over every cached K/V pair. The projection input shrinks from the full `prefix x hidden` matrix to `1 x hidden`.

One common misconception needs to be removed: **KV Cache does not magically change Attention from $O(N^2)$ to $O(N)$.** The new Query still attends over a growing history. The cache removes repeated projection and recomputation of historical K/V vectors; it trades memory for compute. Whether that trade is worthwhile depends on the cache size, which we calculate next.

Two related names will appear later. FlashAttention optimizes memory traffic inside the Attention kernel. PagedAttention manages how KV Cache memory is allocated at the system level. They solve different problems from the decision to keep a KV Cache at all.


## 4. The Memory Cost of KV Cache

The cache occupies GPU memory. A useful estimate is:

$$
\text{KV bytes}
\approx
2 \times L \times T \times H_{kv} \times D \times B \times \text{bytes}
$$

Here, `2` accounts for K and V, `L` is the number of layers, `T` is context length, `H_kv` is the number of KV heads, `D` is head dimension, and `B` is the number of concurrent requests.

Before running code, calculate a realistic 7B-class configuration: 32 layers, 8 GQA KV heads, head dimension 128, batch size 1, context length 8,192, and BF16 values (2 bytes each):

```text
2 (K and V) x 32 (layers) x 8192 (tokens) x 8 (KV heads)
  x 128 (head_dim) x 2 (bytes) x 1 (batch)
= 1,073,741,824 bytes, about 1.07 GB
```

One request with an 8K context needs slightly more than 1 GB. Increasing the context to 128K multiplies this by 16, producing about 17.2 GB—more than the model weights. This is often the first limit encountered by long contexts and concurrent users. The code below verifies the calculation and raises the batch size to 16.


In [ ]:
def kv_cache_gb(layers, tokens, kv_heads, head_dim, batch, bytes_per_elem=2):
    """Estimate KV Cache size in GB from the formula."""
    total = 2 * layers * tokens * kv_heads * head_dim * batch * bytes_per_elem
    return total / 1e9

# Verify the hand calculation: a 7B-class model with 8 GQA KV heads at 8K and 128K context
print(f"8K context, batch 1  : {kv_cache_gb(32, 8192, 8, 128, 1):.2f} GB  (hand calculation: 1.07)")
print(f"128K context, batch 1: {kv_cache_gb(32, 131072, 8, 128, 1):.2f} GB  (hand calculation: 17.2)")
print()

# Compare three KV-head configurations on the same model at batch 16
cfgs = [("MHA 32 KV heads", 32), ("GQA 8 KV heads", 8), ("MQA 1 KV head", 1)]
for name, kvh in cfgs:
    size = kv_cache_gb(32, 8192, kvh, 128, batch=16)
    print(f"{name:<18}: {size:6.2f} GB")

print()
print("Key observation: MHA and MQA differ by 32x, entirely because of the number of KV heads.")


The final column exposes a major systems problem. On the same 24 GB GPU, a GQA model with an 8K context can still hold 16 concurrent requests, while the MHA version may struggle with even four. The model parameters are unchanged; only the number of stored K/V copies differs. The next section explains the resulting 32-fold range.


## 5. MHA, GQA, and MQA

In standard Multi-Head Attention (MHA), each of 32 Query heads has its own K/V head, producing 32 K/V sets. Because cache size is proportional to the number of KV heads, we should ask:

> **Does every Query head need an exclusive copy of K/V?**

Experiments show that it does not. Several Query heads can share one K/V set with little quality loss, while KV Cache size falls by the sharing factor. **GQA** (Grouped-Query Attention) might use 8 KV heads for 32 Query heads, so every four Query heads share one K/V set. At the extreme, **MQA** (Multi-Query Attention) lets all Query heads share a single K/V set.

```text
MHA: 32 Query heads, 32 K/V sets   <- one exclusive set per head
GQA: 32 Query heads,  8 K/V sets   <- one set shared by four heads
MQA: 32 Query heads,  1 K/V set    <- one set shared by all heads
```

MHA to GQA reduces the cache fourfold; MHA to MQA reduces it 32-fold. The saving comes from sharing rather than lower numerical precision. Modern open models such as LLaMA 3 and Qwen commonly use GQA. DeepSeek's MLA goes further by compressing K/V into low-rank vectors, as discussed in the Part 2 chapter on KV Cache architecture. MHA, GQA, MQA, and MLA all answer the same systems question: how many copies of K/V must be stored?


## 6. Decode's Bandwidth Bottleneck

We can now answer the opening question: why does a GPU capable of hundreds of trillions of operations per second generate only tens of tokens per second?

One Decode step for one token requires roughly $2 \times$ parameter count FLOPs, or about $1.4 \times 10^{10}$ FLOPs for a 7B model. That amount of arithmetic is small for a modern GPU. Before calculation begins, however, about 14 GB of weights plus the KV Cache must move from GPU memory into the compute units. At 2 TB/s, moving 14 GB alone takes about 7 ms. Bandwidth therefore caps the rate near 140 tokens per second even while compute utilization remains below 1%.

In short, **Decode is memory-bound: compute units wait for data rather than arithmetic.** Prefill is the opposite. Its large matrices make it primarily compute-bound. This contrast motivates three later optimization paths:

```text
Weights are too large to move each step -> quantization (next chapter)
Only one token is confirmed per step     -> speculative decoding
Many requests share one GPU             -> inference systems
```


In [ ]:
params = 7e9

weight_bf16 = params * 2          # BF16 uses 2 bytes per parameter
weight_int4 = params * 0.5        # INT4 uses 0.5 bytes per parameter
gpu_bw = 2e12                     # 2 TB/s memory bandwidth, illustrative scale

print(f"7B BF16 weights: {weight_bf16/1e9:.0f} GB -> move them in {weight_bf16/gpu_bw*1000:.1f} ms per step")
print(f"7B INT4 weights: {weight_int4/1e9:.1f} GB -> move them in {weight_int4/gpu_bw*1000:.1f} ms per step")
print()
print("Key observation: weights that are 4x smaller take about 4x less time to move per step.")
print("Quantization accelerates Decode by reducing the data moved at every step.")


## Summary

The acceleration techniques in this section revolve around two phases: prefill focuses on saturating GPU compute, while decode focuses on reducing data movement. What we've covered:

- Naive autoregressive generation reprocesses the history prefix repeatedly, with redundant computation growing as O(N²)
- KV Cache stores computed K and V, avoiding recomputation of historical tokens' K/V and hidden states
- KV Cache consumes memory and is read during decode, creating significant memory and bandwidth pressure at long context + large model scale
- MQA/GQA reduces the number of K/V heads, directly shrinking KV Cache
- Model quantization compresses FP16 to INT8/INT4, halving to quartering memory
- GGUF is llama.cpp's standard format, loadable directly by llama-cpp-python
- FlashAttention computes in SRAM without writing intermediate results back to HBM; v2 reorders loops to increase matmul fraction
- PagedAttention manages KV Cache in pages, eliminating fragmentation
- Prefill processes the entire prompt in parallel with no serial dependency between tokens; it is compute-bound; FlashAttention tiling and Prefix Caching are the primary acceleration techniques
- Decode generates tokens one by one, processing only 1 new Q per step; it is memory-bound; KV Cache + GQA + quantization + FlashDecoding are the primary acceleration techniques
- Prefill arithmetic intensity grows linearly with sequence length (S² compute / S memory access); decode arithmetic intensity stays at a low constant
- In decode, Q has only one row; FlashDecoding splits parallelism along the K/V dimension, combining partial results with online softmax
- Continuous Batching dynamically schedules requests for 5-10x throughput improvement

Next section: speculative decoding — using a small model to guess, a large model to verify.


## Exercises

> You can ask an AI to explain concepts, but please run the code and fill in the answers yourself.

**Exercise 1: KV Cache Memory Calculation**

A model has 32 layers, 32 heads, and 128 dimensions per head. Using FP16 storage, generating 4096 tokens. How much memory does the KV Cache for a single request occupy (in MB)?

Hint: `2 x layers x heads x head_dim x seq_len x 2`

### Exercise 1: Estimate KV Cache Size

Turn the estimate into a function: bytes $\approx$ `2 x layers x tokens x KV heads x head_dim x batch x bytes per element`.

Hint: divide by `1e9` when returning GB.


In [ ]:
# Exercise 1: fill in the KV Cache calculation

def kv_cache_gb(layers, tokens, kv_heads, head_dim, batch, bytes_per_elem=2):
    """Return KV Cache size in GB; the factor 2 accounts for both K and V."""
    # TODO: replace the triple-quoted text with your code
    """Compute 2 * layers * tokens * kv_heads * head_dim * batch * bytes_per_elem in GB."""

# 7B-class model, 8 GQA KV heads, 128K context, batch 1, BF16
assert abs(kv_cache_gb(32, 131072, 8, 128, 1) - 17.18) < 0.01
print("✅ Exercise 1 passed: you can now estimate KV Cache size for any configuration.")


### Exercise 2: How Much Do GQA and MQA Save?

KV Cache size is proportional to the number of KV heads. This is the entire source of the memory saving from GQA and MQA.

Hint: divide the KV-head counts of the two configurations to obtain the cache-reduction factor.


In [ ]:
# Exercise 2: fill in KV-head memory ratio

def kv_ratio(kv_heads_a, kv_heads_b):
    """Return the KV Cache size ratio between configurations a and b."""
    # TODO: replace the triple-quoted text with your code
    """KV Cache size is proportional to the number of KV heads, so return a / b."""

assert kv_ratio(32, 8) == 4.0    # MHA -> GQA(8)
assert kv_ratio(32, 1) == 32.0   # MHA -> MQA
print("✅ Exercise 2 passed: you can calculate the memory savings from MHA to GQA to MQA.")


### Exercise 3: Calculate Decode Arithmetic Intensity

For a $4096 \times 4096$ weight matrix, matrix multiplication uses approximately `2 x batch x 4096 x 4096` FLOPs. The weights move from GPU memory once, requiring about `4096 x 4096 x 2` bytes. FLOPs divided by bytes is called arithmetic intensity; it determines whether the GPU waits for computation or data.

Hint: a larger batch performs more arithmetic for the same weight bytes, so arithmetic intensity increases.


In [ ]:
# Exercise 3: fill in arithmetic intensity

def arithmetic_intensity(batch, bytes_per_elem=2):
    """FLOPs divided by weight bytes for one 4096x4096 matrix multiplication."""
    flops = 2 * batch * 4096 * 4096
    weight_bytes = 4096 * 4096 * bytes_per_elem
    # TODO: replace the triple-quoted text with your code
    """Return flops divided by weight_bytes."""

assert arithmetic_intensity(1) == 1.0
assert arithmetic_intensity(64) == 64.0
print("✅ Exercise 3 passed: a larger batch gets more computation from each weight byte, the intuition behind Decode concurrency.")
